### Storm Chandra Presentation 3


In [1]:
import matplotlib.pyplot as plt
import numpy as np

import cartopy.crs as ccrs
import cartopy.feature as cfeature

import rasterio
from rasterio.plot import show
import imageio.v2 as imageio

from pathlib import Path

In [2]:
# coords of target stations

mal_lon, mal_lat = -7.339, 55.372
dublin_airport_lon, dublin_airport_lat = -6.2499, 53.4264
john_lon, john_lat = -6.497, 52.298
val_lon, val_lat = -10.241, 51.938


stations = {
    "Malin Head": (mal_lon, mal_lat, (10, 10)),  # Offset top-right
    "Dublin Airport": (
        dublin_airport_lon,
        dublin_airport_lat,
        (10, 0),
    ),  # Offset right
    "Johnstown Castle": (john_lon, john_lat, (10, -10)),  # Offset bottom-right
    "Valentia Observatory": (
        val_lon,
        val_lat,
        (-10, 10),
    ),  # Offset top-left
}

In [3]:
# function that plots one geotiff file with a map of ireland

def plot_satellite_data(tif, out_path):
    with rasterio.open(tif) as src:
        rgb = np.moveaxis(src.read([1, 2, 3]), 0, -1)
        b = src.bounds
        extent = [b.left, b.right, b.bottom, b.top]

    fig = plt.figure(figsize=(10, 6.5))  # Same size for every frame
    ax = plt.axes(projection=ccrs.PlateCarree())

    # Display GeoTIFF RGB image
    ax.imshow(rgb, origin="upper", extent=extent, transform=ccrs.PlateCarree())

    # Add geographical features
    ax.coastlines(resolution="10m", color="black", linewidth=0.6)
    ax.add_feature(
        cfeature.BORDERS, linestyle=":", edgecolor="black", linewidth=0.6
    )

    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5)
    gl.top_labels = False
    gl.right_labels = False

    # Plot Stations with offsets
    for name, (lon, lat, (dx, dy)) in stations.items():
        # Yellow dot marker
        ax.plot(
            lon,
            lat,
            marker="o",
            markersize=5,
            color="yellow",
            markeredgecolor="black",
            transform=ccrs.PlateCarree(),
            zorder=5,
        )

        # Annotated text with pointer arrow line
        ax.annotate(
            name,
            xy=(lon, lat),
            xycoords=ccrs.PlateCarree()._as_mpl_transform(ax),
            xytext=(dx, dy),
            textcoords="offset points",
            ha="left" if dx >= 0 else "right",
            va="bottom" if dy >= 0 else "top",
            fontsize=9,
            fontweight="bold",
            color="white",
            bbox=dict(
                boxstyle="round,pad=0.2", fc="black", ec="none", alpha=0.6
            ),
            arrowprops=dict(arrowstyle="-", color="white", lw=0.8),
            zorder=6,
        )

    ax.set_title(Path(tif).stem)
    fig.savefig(out_path, dpi=150)  # Preserves exact canvas size for GIF
    plt.close(fig)

In [4]:
# loop over the folder and make plots

tif_dir = Path(
'/home/ellen/Downloads/AIMSIR Semester 1/Physical Meteorology/phys-met/geoTIFF data')

out_dir = Path(
'/home/ellen/Downloads/AIMSIR Semester 1/Physical Meteorology/phys-met/output satellite plots')

out_dir.mkdir(parents=True, exist_ok=True)

tif_files = sorted(tif_dir.glob("*.tif"))   # use "*.tif*" to also catch .tiff

for tif in tif_files:
    out_path = out_dir / f"{tif.stem}.png"
    try:
        plot_satellite_data(tif, out_path)
        print(f"Saved {out_path}")
    except Exception as e:
        print(f"Failed on {tif.name}: {e}")

Saved /home/ellen/Downloads/AIMSIR Semester 1/Physical Meteorology/phys-met/output satellite plots/storm_chandra_track_2026-01-26T2100.png
Saved /home/ellen/Downloads/AIMSIR Semester 1/Physical Meteorology/phys-met/output satellite plots/storm_chandra_track_2026-01-26T2200.png
Saved /home/ellen/Downloads/AIMSIR Semester 1/Physical Meteorology/phys-met/output satellite plots/storm_chandra_track_2026-01-26T2300.png
Saved /home/ellen/Downloads/AIMSIR Semester 1/Physical Meteorology/phys-met/output satellite plots/storm_chandra_track_2026-01-27T0000.png
Saved /home/ellen/Downloads/AIMSIR Semester 1/Physical Meteorology/phys-met/output satellite plots/storm_chandra_track_2026-01-27T0100.png
Saved /home/ellen/Downloads/AIMSIR Semester 1/Physical Meteorology/phys-met/output satellite plots/storm_chandra_track_2026-01-27T0200.png
Saved /home/ellen/Downloads/AIMSIR Semester 1/Physical Meteorology/phys-met/output satellite plots/storm_chandra_track_2026-01-27T0300.png
Saved /home/ellen/Downloads

In [5]:
# now make the plots into a gif
  
frames = [imageio.imread(p) for p in sorted(out_dir.glob("*.png"))]
imageio.mimsave(out_dir / "storm_chandra_RGB.gif", frames, duration=400, loop=0)   # ~2.5 fps